# Trying out numpy to OpenFOAM format conversion:

In [3]:
import numpy as np
from pathlib import Path
import json
import Ofpp

In [4]:
u = np.load("/home/ninelab/repitframework/Assets/natural_convection/U_3.npy")

In [1]:
from pathlib import Path
import numpy as np
import struct

# Other functions like parse_internal_field, parse_boundary_content, etc., remain as they are

def write_field(fn, internal_field=None, boundary_field=None, binary=False):
    """
    Write data back to OpenFOAM dictionary file with numpy array support.
    :param fn: File name to write
    :param internal_field: numpy array for the internal field
    :param boundary_field: dict of numpy arrays for each boundary
    :param binary: Whether to write in binary format
    """
    content = []
    if internal_field is not None:
        content += write_internal_field(internal_field, binary)

    if boundary_field is not None:
        content += write_boundary_field(boundary_field, binary)

    with open(fn, "wb") as f:
        f.writelines(content)

def write_internal_field(internal_field, binary=False):
    """
    Convert internal field numpy array to OpenFOAM dictionary format
    :param internal_field: numpy array
    :param binary: binary format or not
    :return: formatted string as a list of bytes lines
    """
    if internal_field.ndim == 1:
        field_str = "internalField nonuniform List<scalar> {}\n".format(len(internal_field))
    else:
        field_str = "internalField nonuniform List<vector> {}\n".format(len(internal_field))

    lines = [field_str.encode()]
    
    if binary:
        header = b"("
        footer = b");\n"
        buf = struct.pack('c{}d'.format(internal_field.size), b'\0', *internal_field.flatten())
        lines.extend([header, buf, footer])
    else:
        lines.append(b"(\n")
        for value in internal_field:
            line = " ".join(map(str, value)) if internal_field.ndim > 1 else str(value)
            lines.append(f"{line}\n".encode())
        lines.append(b");\n")
    
    return lines

def write_boundary_field(boundary_field, binary=False):
    """
    Convert boundary field dictionary to OpenFOAM dictionary format
    :param boundary_field: dict of numpy arrays for each boundary
    :param binary: binary format or not
    :return: formatted string as a list of bytes lines
    """
    lines = [b"boundaryField\n{\n"]

    for boundary, field_data in boundary_field.items():
        lines.append(f"  {boundary.decode()} {{\n".encode())

        if isinstance(field_data, np.ndarray):
            field_lines = write_internal_field(field_data, binary=binary)
            lines.extend([b"    " + line for line in field_lines])
        elif isinstance(field_data, dict):
            for key, value in field_data.items():
                if key == "uniform":
                    lines.append(f"    {key} uniform {value};\n".encode())
                elif key == "nonuniform":
                    lines.extend(write_internal_field(np.array(value), binary=binary))
        
        lines.append(b"  }\n")

    lines.append(b"}\n")
    return lines


In [5]:
u.ndim

2